# Phase 4A1 — TPU Version: Stride-2 and Max-Pooling RG Validation
## ThermoRG v8: Validate beta = beta0 * phi^n_s with phi = 0.87

**TPU-optimized version** for Kaggle TPU runtime.

| Name | n_s | Type |
|------|-----|------|
| ThermoNet-L3 | 0 | Baseline |
| ThermoNet-S2-1 | 1 | stride-2 |
| ThermoNet-S2-2 | 2 | stride-2 |
| ThermoNet-MP-2 | 2 | max-pool |

## Step 1: TPU Setup

In [ ]:
# TPU setup
import os
# Kaggle TPU: runtime type is set via Accelerator, not env var
# Check if torch_xla is available and TPU runtime is active
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    DEVICE = xm.xla_device()
    print("TPU device:", DEVICE)
    print("TPU runtime: OK")
except Exception as e:
    print(f"TPU setup failed: {e}")
    print("Make sure Accelerator is set to TPU in notebook settings")


## Step 2: Clone + Install

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned")

## Step 3: Imports

In [ ]:
import sys, json, math
import numpy as np
import torch
import torch.nn as nn
from torch import linalg
from pathlib import Path
import torch_xla.core.xla_model as xm

sys.path.insert(0, '/kaggle/working/ThermoRG-NN')
sys.path.insert(0, '/kaggle/working/ThermoRG-NN/experiments/lift_test')

DEVICE = xm.xla_device()
print("Device:", DEVICE)

## Step 4: J_topo Computation

In [ ]:
def compute_D_eff(W):
    if W.dim() == 4: W = W.view(W.size(0), -1)
    W_cpu = W.cpu().float()
    fro_sq = (W_cpu ** 2).sum().item()
    S = linalg.svd(W_cpu).S
    return fro_sq / (S[0].item() ** 2 + 1e-12)

def compute_J_topo(model):
    skip_excl = ['layernorm', 'norm', 'batchnorm', 'bn', 'flatten', 'fc', 'linear']
    eta_list, prev_D, prev_C = [], None, None
    for name, m in model.named_modules():
        if any(p in name.lower() for p in skip_excl): continue
        if isinstance(m, nn.Conv2d):
            W, s = m.weight.data.float(), m.stride[0]
            cin, cout = m.in_channels, m.out_channels
            D = compute_D_eff(W)
            if prev_D and prev_C:
                eta = D/prev_D
                if s > 1: eta *= (cout/prev_C)/(s**2)
                eta_list.append(float(eta))
            prev_D, prev_C = D, cout
        elif isinstance(m, nn.MaxPool2d):
            s = m.stride if hasattr(m,'stride') else 2
            if s > 1 and prev_D:
                eta_list.append(1.0/(s**2))
    if not eta_list: return 1.0, []
    return math.exp(-np.mean([abs(math.log(max(e,1e-10))) for e in eta_list])), eta_list

def count_ds(model):
    nc = nm = 0
    for m in model.modules():
        if isinstance(m, nn.Conv2d) and m.stride[0]==2: nc+=1
        elif isinstance(m, nn.MaxPool2d): nm+=1
    return nc, nm

## Step 5: Define Architectures

In [ ]:
class L3(nn.Module):
    def __init__(self,ch=64):
        super().__init__()
        bl=[nn.Conv2d(3,ch,3,padding=1,bias=False),nn.BatchNorm2d(ch),nn.GELU(),
            nn.Conv2d(ch,ch*2,3,padding=1,bias=False),nn.BatchNorm2d(ch*2),nn.GELU(),
            nn.Conv2d(ch*2,ch*2,3,padding=1,bias=False),nn.BatchNorm2d(ch*2),nn.GELU()]
        self.bl=nn.ModuleList(bl)
        self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.fc=nn.Linear(ch*2,10)
    def forward(self,x):
        for b in self.bl: x=b(x)
        return self.fc(self.pool(x).view(x.size(0),-1))

class S2_1(nn.Module):
    def __init__(self,ch=64):
        super().__init__()
        self.c1=nn.Conv2d(3,ch,3,padding=1,bias=False); self.b1=nn.BatchNorm2d(ch); self.a1=nn.GELU()
        self.d1=nn.Conv2d(ch,ch*2,3,stride=2,padding=1,bias=False); self.bd1=nn.BatchNorm2d(ch*2); self.ad1=nn.GELU()
        self.c2=nn.Conv2d(ch*2,ch*2,3,padding=1,bias=False); self.b2=nn.BatchNorm2d(ch*2); self.a2=nn.GELU()
        self.pool=nn.AdaptiveAvgPool2d((1,1)); self.fc=nn.Linear(ch*2,10)
    def forward(self,x):
        x=self.a1(self.b1(self.c1(x))); x=self.ad1(self.bd1(self.d1(x))); x=self.a2(self.b2(self.c2(x)))
        return self.fc(self.pool(x).view(x.size(0),-1))

class S2_2(nn.Module):
    def __init__(self,ch=64):
        super().__init__()
        self.c1=nn.Conv2d(3,ch,3,padding=1,bias=False); self.b1=nn.BatchNorm2d(ch); self.a1=nn.GELU()
        self.d1=nn.Conv2d(ch,ch*2,3,stride=2,padding=1,bias=False); self.bd1=nn.BatchNorm2d(ch*2); self.ad1=nn.GELU()
        self.c2=nn.Conv2d(ch*2,ch*2,3,padding=1,bias=False); self.b2=nn.BatchNorm2d(ch*2); self.a2=nn.GELU()
        self.d2=nn.Conv2d(ch*2,ch*4,3,stride=2,padding=1,bias=False); self.bd2=nn.BatchNorm2d(ch*4); self.ad2=nn.GELU()
        self.c3=nn.Conv2d(ch*4,ch*4,3,padding=1,bias=False); self.b3=nn.BatchNorm2d(ch*4); self.a3=nn.GELU()
        self.pool=nn.AdaptiveAvgPool2d((1,1)); self.fc=nn.Linear(ch*4,10)
    def forward(self,x):
        x=self.a1(self.b1(self.c1(x))); x=self.ad1(self.bd1(self.d1(x)))
        x=self.a2(self.b2(self.c2(x))); x=self.ad2(self.bd2(self.d2(x)))
        return self.fc(self.pool(self.a3(self.b3(self.c3(x)))).view(x.size(0),-1))

class MP_2(nn.Module):
    def __init__(self,ch=64):
        super().__init__()
        self.c1=nn.Conv2d(3,ch,3,padding=1,bias=False); self.b1=nn.BatchNorm2d(ch); self.a1=nn.GELU()
        self.p1=nn.MaxPool2d(2)
        self.c2=nn.Conv2d(ch,ch*2,3,padding=1,bias=False); self.b2=nn.BatchNorm2d(ch*2); self.a2=nn.GELU()
        self.p2=nn.MaxPool2d(2)
        self.c3=nn.Conv2d(ch*2,ch*4,3,padding=1,bias=False); self.b3=nn.BatchNorm2d(ch*4); self.a3=nn.GELU()
        self.pool=nn.AdaptiveAvgPool2d((1,1)); self.fc=nn.Linear(ch*4,10)
    def forward(self,x):
        x=self.a1(self.b1(self.c1(x))); x=self.p1(x)
        x=self.a2(self.b2(self.c2(x))); x=self.p2(x)
        return self.fc(self.pool(self.a3(self.b3(self.c3(x)))).view(x.size(0),-1))

ARCHS = {
    'ThermoNet-L3':('baseline',L3()),
    'ThermoNet-S2-1':('stride2',S2_1()),
    'ThermoNet-S2-2':('stride2',S2_2()),
    'ThermoNet-MP-2':('maxpool',MP_2()),
}

for n,(t,a) in ARCHS.items():
    a.eval()
    J,_=compute_J_topo(a)
    nc,nm=count_ds(a)
    print(n, 'n_s=', nc+nm, 'J_topo=', J)

## Step 6: Compute Predictions

In [ ]:
beta0, phi_val = 0.86, 0.87
preds = {}
for n,(t,a) in ARCHS.items():
    nc,nm=count_ds(a)
    ns=nc+nm
    bp=beta0*(phi_val**ns)
    J,_=compute_J_topo(a)
    preds[n]={'type':t,'n_s':ns,'J_topo':J,'beta_pred':bp}
    print(n, 'beta_pred=', bp)

## Step 7: Training Setup

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader,Subset

D_VALS=[10000,25000]
EPOCHS=50
BATCH_SIZE=256

TFM=T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),T.ToTensor(),
               T.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])])
TFMV=T.Compose([T.ToTensor(),T.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])])
train_ds=CIFAR10(root='./data',train=True,transform=TFM,download=True)
val_ds=CIFAR10(root='./data',train=False,transform=TFMV,download=False)
val_loader=DataLoader(val_ds,batch_size=512,shuffle=False,num_workers=4)

def get_loader(D):
    return DataLoader(Subset(train_ds,list(range(D))),batch_size=BATCH_SIZE,shuffle=True,num_workers=4)

## Step 8: Training (TPU-optimized)

In [ ]:
def train(model,dloader,vloader,ep,lr,wd):
    model=model.to(DEVICE)
    opt=torch.optim.SGD(model.parameters(),lr=lr,momentum=0.9,weight_decay=wd,nesterov=True)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=ep)
    crit=nn.CrossEntropyLoss()
    best_loss,best_acc,best_ep=float('inf'),0.0,0
    for e in range(ep):
        model.train()
        for X,y in dloader:
            X,y=X.to(DEVICE),y.to(DEVICE)
            opt.zero_grad()
            loss=crit(model(X),y)
            loss.backward()
            xm.optimizer_step(opt,barrier=True)
        sched.step()
        model.eval()
        loss_sum,acc_sum,total=0,0,0
        with torch.no_grad():
            for X,y in vloader:
                X,y=X.to(DEVICE),y.to(DEVICE)
                o=model(X)
                loss_sum+=crit(o,y).item()*X.size(0)
                acc_sum+=(o.argmax(1)==y).sum().item(); total+=X.size(0)
        loss_val=loss_sum/total
        acc_val=acc_sum/total
        if loss_val<best_loss: best_loss=loss_val; best_acc=acc_val; best_ep=e+1
        if e%10==0: print('Epoch',e,'loss=',round(loss_val,4),'acc=',round(acc_val,4))
    return {'best_val_loss':best_loss,'best_val_acc':best_acc,'best_epoch':best_ep}

## Step 9: Run Experiment

In [ ]:
from scipy.optimize import curve_fit

def fit(losses,D_vals):
    try:
        popt,_=curve_fit(lambda D,a,b,e: a*np.array(D)**(-b)+e,D_vals,
                        [losses[d] for d in D_vals],p0=[10,0.5,0.5],bounds=([0,0,0],[1000,5,10]),maxfev=10000)
        R2=1-((np.array([losses[d] for d in D_vals])-popt[0]*np.array(D_vals)**(-popt[1])-popt[2])**2).sum()\
            /((np.array([losses[d] for d in D_vals])-np.mean([losses[d] for d in D_vals]))**2).sum()
        return {'alpha':popt[0],'beta':popt[1],'E':popt[2],'R2':R2}
    except: return {'alpha':None,'beta':None,'E':None,'R2':0}

all_res={}
for name,(atype,tmpl) in ARCHS.items():
    print('\n',name)
    res={'D_results':{},'J_topo':None,'n_s':None}
    tmpl.eval()
    J,_=compute_J_topo(tmpl)
    nc,nm=count_ds(tmpl)
    ns=nc+nm
    res['J_topo']=J; res['n_s']=ns; res['beta_pred']=preds[name]['beta_pred']
    losses={}
    for D in D_VALS:
        print(' D=',D,end='',flush=True)
        if name=='ThermoNet-L3': m=L3().to(DEVICE)
        elif name=='ThermoNet-S2-1': m=S2_1().to(DEVICE)
        elif name=='ThermoNet-S2-2': m=S2_2().to(DEVICE)
        elif name=='ThermoNet-MP-2': m=MP_2().to(DEVICE)
        r=train(m,get_loader(D),val_loader,EPOCHS,0.01,5e-4)
        losses[D]=r['best_val_loss']
        res['D_results'][str(D)]=r
        print(' loss=',round(r['best_val_loss'],4))
        del m
    fit_r=fit(losses,D_VALS)
    res['scaling_fit']=fit_r
    print(' beta_fit=',fit_r.get('beta'),'R2=',fit_r.get('R2'))
    all_res[name]=res
print('\nDone!')

## Step 10: Analysis

In [ ]:
print('\nRESULTS:')
errors=[]
for n,r in all_res.items():
    bp=r['beta_pred']
    bo=r['scaling_fit'].get('beta')
    err=bo-bp if bo else None
    if err: errors.append(err)
    print(n,'pred=',round(bp,4),'obs=',round(bo,4),'err=',round(err,4) if err else 'N/A')
if errors:
    rmse=math.sqrt(np.mean(np.array(errors)**2))
    print('RMSE=',round(rmse,4),'PASS' if rmse<0.05 else 'FAIL')

## Step 11: Save

In [ ]:
out=Path('./phase_4a1_tpu_results'); out.mkdir()
def ms(obj):
    if isinstance(obj,dict): return {k:ms(v) for k,v in obj.items()}
    return float(obj) if isinstance(obj,(np.floating,np.integer)) else list(obj) if isinstance(obj,np.ndarray) else obj
with open(out/'results.json','w') as f: json.dump(ms(all_res),f,indent=2)
print('Saved to',out/'results.json')

## Step 12: Git Push

In [ ]:
%cd /kaggle/working/ThermoRG-NN
!git add -A && git commit -m "Phase 4A1 TPU results" && git push